In [75]:
import serpapi
import pandas as pd
import math

In [81]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lam = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lam / 2) ** 2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError

_geolocator = Nominatim(user_agent="flight_bus_notebook")

def geocode_city(city: str) -> tuple[float, float]:
    """Return (latitude, longitude) for a city name.

    Raises ValueError if the city cannot be geocoded.
    """
    try:
        location = _geolocator.geocode(city, exactly_one=True, timeout=10)
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        raise ValueError(f"Geocoding service error for '{city}': {e}") from e

    if location is None:
        raise ValueError(f"Could not geocode city: '{city}'. Check the spelling or try a more specific name.")

    return location.latitude, location.longitude


In [80]:
def explore_search(departure_id, outbound_date, target_city: str, MAX_DISTANCE_KM=300):
    lat, lon = geocode_city(target_city)  # raises ValueError if city not found
    TARGET = {"city": target_city, "latitude": lat, "longitude": lon}

    params = {
        "engine": "google_travel_explore",
        "departure_id": departure_id,
        "currency": "EUR",
        "type": "2",
        "outbound_date": outbound_date,
        "no_cache": False,
    }
    client = serpapi.Client(api_key=api_key)
    results = client.search(params)

    rows = []
    for flight in results["destinations"]:
        if "flight_price" not in flight:
            continue
        rows.append({
            "city":      flight["name"],
            "country":   flight["country"],
            "latitude":  flight["gps_coordinates"]["latitude"],
            "longitude": flight["gps_coordinates"]["longitude"],
            "price":     flight["flight_price"],
            "duration":  flight["flight_duration"],
            "airline":   flight["airline"],
            "link":      flight["link"],
        })

    df = pd.DataFrame(rows).sort_values("price").reset_index(drop=True)
    df["distance_to_target_km"] = df.apply(
        lambda row: haversine_km(row["latitude"], row["longitude"], TARGET["latitude"], TARGET["longitude"]),
        axis=1,
    )

    nearby = (
        df[df["distance_to_target_km"] <= MAX_DISTANCE_KM]
        [["city", "country", "price", "duration", "airline", "latitude", "longitude"]]
        .sort_values("price")
        .reset_index(drop=True)
    )
    return nearby


In [58]:
def flight_search(departure_id, arrival_id, outbound_date):
    client = serpapi.Client(api_key=api_key)
    results = client.search({
        "engine": "google_flights",
        "departure_id": departure_id,
        "arrival_id": arrival_id,
        "currency": "EUR",
        "type": "2",
        "outbound_date": outbound_date,
        "no_cache": False,
    })
    best_flights = results["best_flights"]
    other_flights = results["other_flights"]
    rows = []
    for flight in other_flights:
        rows.append({
            "airlane":   flight["flights"][0]["airline"],
            "price":     flight["price"],
            "departure_time": flight["flights"][0]["departure_airport"]["time"],
            "departure_iata": flight["flights"][0]["departure_airport"]["id"],
            "arrival_time": flight["flights"][0]["arrival_airport"]["time"],
            "arrival_iata": flight["flights"][0]["arrival_airport"]["id"],
            "duration":  flight["total_duration"],
        })

    other_flights_df = pd.DataFrame(rows)

    rows = []
    for flight in best_flights:
        rows.append({
            "airlane":   flight["flights"][0]["airline"],
            "price":     flight["price"],
            "departure_time": flight["flights"][0]["departure_airport"]["time"],
            "departure_iata": flight["flights"][0]["departure_airport"]["id"],
            "arrival_time": flight["flights"][0]["arrival_airport"]["time"],
            "arrival_iata": flight["flights"][0]["arrival_airport"]["id"],
            "duration":  flight["total_duration"],
        })

    best_flights_df = pd.DataFrame(rows)
    return other_flights_df,best_flights_df


In [48]:
from checkmybus import CheckMyBusClient, CheckMyBusSearchParams

def bus_train_transfer(departure_location= "frankfurt am main, Germany", arrival_location = "Munich, Germany",                                       departure_date="2026-05-16"
                       ):
    client = CheckMyBusClient()
    result = client.search(CheckMyBusSearchParams(
    departure_location=departure_location,
    departure_date=departure_date,
    arrival_location=arrival_location,
    ))

    df = result.to_dataframe()
    return df


# Uçak + otobos ve tren kodları

In [110]:
# travel_planner.py
import math
import json
from functools import lru_cache
from dataclasses import dataclass
import pandas as pd
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderServiceError
import os
import serpapi
from dotenv import load_dotenv
# ========================= CONFIG =========================
load_dotenv()
API_KEY = os.getenv("SERPAPI_KEY")  # ← put in .env, never hardcode
GEOLOCATOR = Nominatim(user_agent="travel_planner_kerem")  # ONE instance

COUNTRY_EN = {"DE": "Germany", "IT": "Italy"}  # extend as needed

# ========================= DATA =========================
with open("../airports.json") as f:  # put json next to script or use absolute path
    airports_df = (
        pd.DataFrame.from_dict(json.load(f), orient="index")
        .query("country in ['DE', 'IT'] and iata != ''")
        [["iata", "name", "city", "country", "lat", "lon"]]
        .reset_index(drop=True)
    )

# ========================= HELPERS =========================
@dataclass
class Location:
    city: str
    country: str
    lat: float
    lon: float
    iata: str | None = None
    bus_name: str | None = None  # "City, Country" for CheckMyBus

@lru_cache(maxsize=100)  # ← caches repeated geocoding calls!
def geocode_city(city: str) -> tuple[float, float]:
    try:
        loc = GEOLOCATOR.geocode(city, exactly_one=True, timeout=10)
        if loc is None:
            raise ValueError(f"Could not geocode '{city}'")
        return loc.latitude, loc.longitude
    except (GeocoderTimedOut, GeocoderServiceError) as e:
        raise ValueError(f"Geocoding error for '{city}': {e}") from e

@lru_cache(maxsize=100)
def city_en_from_coords(lat: float, lon: float, country_code: str, iata: str = "") -> str:
    """Reverse geocode → perfect CheckMyBus string (English)."""
    country = COUNTRY_EN.get(country_code, country_code)
    loc = GEOLOCATOR.reverse((lat, lon), language="en", addressdetails=True, exactly_one=True, timeout=10)
    if not loc:
        raise ValueError(f"Reverse geocode failed for ({lat}, {lon})")
    addr = loc.raw.get("address", {})
    city = addr.get("city") or addr.get("town") or addr.get("village") or addr.get("county")
    if not city:
        raise ValueError(f"No city in address for ({lat}, {lon})")
    return f"{city}, {country}"

def iata_to_location(iata: str) -> Location:
    row = airports_df[airports_df["iata"] == iata.upper()]
    if row.empty:
        raise ValueError(f"IATA {iata} not in airports.json")
    r = row.iloc[0]
    country = COUNTRY_EN.get(r["country"], r["country"])
    bus_name = f'{r["city"]}, {country}'
    return Location(city=r["city"], country=country, lat=r["lat"], lon=r["lon"], iata=iata, bus_name=bus_name)

# ========================= FLIGHTS =========================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1 = math.radians(lat1)
    phi2 = math.radians(lat2)
    d_phi = math.radians(lat2 - lat1)
    d_lam = math.radians(lon2 - lon1)
    a = math.sin(d_phi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(d_lam / 2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

def explore_search(departure_id: str, outbound_date: str, target_city: str, max_distance_km: int = 300):
    """Cheap flights to cities near your target + IATA + bus-ready name."""
    lat, lon = geocode_city(target_city)
    target = {"lat": lat, "lon": lon}

    client = serpapi.Client(api_key=API_KEY)
    results = client.search({
        "engine": "google_travel_explore",
        "departure_id": departure_id,
        "currency": "EUR",
        "type": "2",
        "outbound_date": outbound_date,
        "no_cache": False,
    })

    rows = []
    for dest in results.get("destinations", []):
        if "flight_price" not in dest:
            continue
        iata = dest.get("destination_airport", {}).get("code")  # ← NEW!
        lat_d, lon_d = dest["gps_coordinates"]["latitude"], dest["gps_coordinates"]["longitude"]
        distance = haversine_km(lat_d, lon_d, target["lat"], target["lon"])

        if distance > max_distance_km:
            continue

        # Build clean bus name (prefer reverse geocode on GPS)
        try:
            bus_name = city_en_from_coords(lat_d, lon_d, country_code=dest.get("country", "")[:2] or "DE", iata=iata or "")
        except Exception:
            # fallback
            bus_name = f"{dest['name']}, {dest.get('country', 'Unknown')}"

        rows.append({
            "city": dest["name"],
            "country": dest.get("country"),
            "iata": iata,
            "price": dest["flight_price"],
            "duration": dest["flight_duration"],
            "airline": dest["airline"],
            "link": dest["link"],
            "lat": lat_d,
            "lon": lon_d,
            "distance_km": round(distance, 1),
            "bus_departure_name": bus_name,   # ← ready for CheckMyBus
        })

    df = pd.DataFrame(rows).sort_values("price").reset_index(drop=True)
    return df

# ========================= BUS / TRAIN =========================
from checkmybus import CheckMyBusClient, CheckMyBusSearchParams

def bus_train_transfer(departure_location: str, arrival_location: str, departure_date: str):
    """Return DataFrame of buses/trains. departure_location must be clean 'City, Country'."""
    client = CheckMyBusClient()

    print(f"🔎 Bus/Train search STARTING: {departure_location} → {arrival_location} on {departure_date}")

    try:
        search_result = client.search(CheckMyBusSearchParams(
            departure_location=departure_location,
            arrival_location=arrival_location,
            departure_date=departure_date,
        ))

        df = search_result.to_dataframe()          # renamed for clarity

        # Count buses and trains safely (in case column is missing)
        if "transport_mode" in df.columns:
            count_bus = len(df[df["transport_mode"] == "bus"])
            count_train = len(df[df["transport_mode"] == "train"])
        else:
            count_bus = count_train = 0
            print("⚠️  'transport_mode' column not found in result!")

        print(f"✅ Search DONE for {departure_location}. "
              f"Total: {len(df)} results → {count_bus} bus(es), {count_train} train(s)")

        if df.empty:
            print(f"⚠️  No buses or trains found from {departure_location} to {arrival_location}")

        return df

    except Exception as e:   # catches API errors, bad parameters, network issues, etc.
        print(f"❌ ERROR searching {departure_location} → {arrival_location}: {e}")
        # You can raise or return empty DataFrame depending on how strict you want it
        return pd.DataFrame()   # empty DF so the main pipeline doesn't crash

# ========================= MAIN PIPELINE (the simple way) =========================
def find_cheap_flight_plus_ground(departure_id: str,
                                  target_city: str,          # e.g. "Munich, Germany" or just "Munich"
                                  outbound_date: str,        # "2026-05-16"
                                  ground_date: str = None,   # defaults to same day
                                  max_distance_km: int = 300):
    if ground_date is None:
        ground_date = outbound_date

    # 1. Get cheap flights near target
    nearby = explore_search(departure_id, outbound_date, target_city, max_distance_km)
    if nearby is not None:
        print(f"Hey good news! Found {len(nearby)} nearby flights.")
    else:
        print("Oh sorry! There is no nearby flights.")
    for _, row in nearby.iterrows():
        print(f"From {row["city"]} and price is {row["price"]}")

    # 2. For each, search ground transport
    combined = []
    for _, row in nearby.iterrows():
        try:
            ground_df = bus_train_transfer(                  # I renamed the variable to "ground_df" for clarity
                departure_location=row["bus_departure_name"],
                arrival_location=target_city,
                departure_date=ground_date
            )
            if ground_df.empty or "price" not in ground_df.columns:
                continue

            # ── NEW: handle BOTH bus and train in one clean loop ──
            for mode in ["bus", "train"]:
                # Filter the right transport mode (safe if column doesn't exist yet)
                mode_df = ground_df[ground_df.get("transport_mode", pd.Series(["bus"] * len(ground_df))) == mode]

                if not mode_df.empty:
                    best = mode_df.loc[mode_df["price"].idxmin()]   # cheapest in this mode

                    combined.append({
                        **row.to_dict(),                     # all flight info
                        "ground_type": mode.capitalize(),    # "Bus" or "Train"
                        "ground_price": best["price"],
                        "total_price": row["price"] + best["price"],
                        "ground_duration": best.get("duration_min", best.get("duration", None)),
                        "ground_link": best.get("checkmybus_url", best.get("link", None)),
                        "ground_departure": best.get("departure_dt", best.get("departure_time", None)),
                        "ground_arrival": best.get("arrival_dt", best.get("arrival_time", None)),
                    })
        except Exception as e:
            print(f"Bus search failed for {row['city']}: {e}")  # or log

    if combined:
        result_df = pd.DataFrame(combined).sort_values("total_price").reset_index(drop=True)
        return result_df[["city", "iata", "price", "ground_type", "ground_price", "total_price",
                          "distance_km", "bus_departure_name", "link",
                          "ground_duration", "ground_departure", "ground_arrival"]]
    else:
        return pd.DataFrame()  # empty



In [111]:
import os
import pandas as pd
from serpapi import Client as SerpApiClient  # clearer import

def flight_search(departure_id: str, arrival_id: str, outbound_date: str):
    """
    Search direct/specific flights from departure_id → arrival_id.
    Returns ONE DataFrame with both 'Best' and 'Other' flights.
    """
    api_key = os.getenv("SERPAPI_KEY")
    if not api_key:
        raise ValueError("SERPAPI_KEY environment variable not set!")

    print(f"✈️  Flight search STARTING: {departure_id} → {arrival_id} on {outbound_date}")

    try:
        client = SerpApiClient(api_key=api_key)
        results = client.search({
            "engine": "google_flights",
            "departure_id": departure_id.upper(),
            "arrival_id": arrival_id.upper(),
            "currency": "EUR",
            "type": "2",           # round-trip? change if you want one-way
            "outbound_date": outbound_date,
            "no_cache": False,
        })

        best_flights = results.get("best_flights", [])
        other_flights = results.get("other_flights", [])

        rows = []

        def extract_flight(flight_data, flight_type: str):
            # Safely handle possible multi-leg flights
            leg = flight_data["flights"][0] if flight_data.get("flights") else {}
            return {
                "flight_type": flight_type,
                "airline": leg.get("airline", "Unknown"),
                "price": flight_data.get("price"),
                "departure_time": leg.get("departure_airport", {}).get("time"),
                "departure_iata": leg.get("departure_airport", {}).get("id"),
                "arrival_time": leg.get("arrival_airport", {}).get("time"),
                "arrival_iata": leg.get("arrival_airport", {}).get("id"),
                "duration": flight_data.get("total_duration"),
                "stops": flight_data.get("stops", 0),                    # new
                "flight_number": leg.get("flight_number", ""),           # new
                "link": flight_data.get("link"),                         # booking link if present
            }

        # Best flights
        for f in best_flights:
            rows.append(extract_flight(f, "Best"))

        # Other flights
        for f in other_flights:
            rows.append(extract_flight(f, "Other"))

        df = pd.DataFrame(rows)

        if df.empty:
            print(f"⚠️  No flights found from {departure_id} to {arrival_id}")
        else:
            print(f"✅ Flight search DONE: {len(df)} flights found "
                  f"({len(df[df['flight_type']=='Best'])} best)")

        return df.sort_values("price").reset_index(drop=True)

    except Exception as e:
        print(f"❌ ERROR in flight_search {departure_id} → {arrival_id}: {e}")
        return pd.DataFrame()   # return empty so your main pipeline never crashes

In [112]:
df = flight_search(departure_id="VCE", arrival_id="NUE", outbound_date="2026-05-10")

✈️  Flight search STARTING: VCE → NUE on 2026-05-10
✅ Flight search DONE: 26 flights found (3 best)


In [113]:
df

,flight_type,airline,price,departure_time,departure_iata,arrival_time,arrival_iata,duration,stops,flight_number,link
0,Other,Air Serbia,240,2026-05-10 20:30,VCE,2026-05-10 22:25,BEG,1415,0,JU 465,None
1,Other,Lufthansa,270,2026-05-10 06:30,VCE,2026-05-10 08:00,FRA,900,0,LH 333,None
2,Other,Lufthansa,270,2026-05-10 06:30,VCE,2026-05-10 08:00,FRA,900,0,LH 333,None
3,Other,Lufthansa,286,2026-05-10 06:30,VCE,2026-05-10 08:00,FRA,960,0,LH 333,None
4,Other,Air France,320,2026-05-10 05:55,VCE,2026-05-10 07:50,CDG,980,0,AF 1327,None
5,Best,KLM,398,2026-05-10 06:00,VCE,2026-05-10 07:50,AMS,460,0,KL 1628,None
6,Other,Eurowings,418,2026-05-10 15:45,VCE,2026-05-10 17:15,CGN,1080,0,EW 815,None
7,Other,Eurowings,418,2026-05-10 15:45,VCE,2026-05-10 17:15,CGN,1080,0,EW 815,None
8,Other,Air France,434,2026-05-10 05:55,VCE,2026-05-10 07:50,CDG,720,0,AF 1327,None
9,Other,Lufthansa,455,2026-05-10 06:30,VCE,2026-05-10 08:00,FRA,675,0,LH 333,None


In [ ]:
explore_search()